# Notebook 3 Lab: Continuous Representations, Edges, Color, and Tone

This lab explores continuous image representations through interactive experiments. The reusable implementations live in `continuous.py`. You are invited to inspect the code yourself or ask an LLM to explain, install, or modify pieces of the notebook. Your job is to run experiments, change parameters, use your own images, find failure cases, and explain what assumptions were used or violated.

This notebook is a lab, not a tutorial. The algorithms are already implemented. Your task is to build intuition by making things happen on purpose.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display

from image_utils import load_image, show
from continuous import (
    apply_color_cast,
    as_float_image,
    canny_widget,
    color_space_widget,
    image_chromaticity_widget,
    derivative_clean_widget,
    derivative_gradient_reveal_widget,
    derivative_noise_widget,
    directional_derivative_widget,
    gamma_tone_widget,
    gaussian_blur,
    gamma_decode,
    gamma_encode,
    interpolation_1d_widget,
    interpolation_2d_widget,
    rgb_to_gray,
    signal_sampling_widget,
    simulate_underwater,
    tone_map,
    white_balance_widget,
)

plt.rcParams["figure.dpi"] = 110

## 1. Continuous Signals and Samples

A digital image is a grid of samples from an underlying visual signal. The samples are numbers; the continuous signal between them is an assumption we make.

The blue curve is the chosen cosine signal. The red points are the measured samples. The three curves on the right are candidate cosine signals with the same phase but different frequencies.

Exploration missions:

- Can you tune the three candidate frequencies so their curves pass through the red samples?
- What frequencies work when `samples` is small?
- What frequencies stop working as you increase `samples`?
- Can you find a relation between the number of samples, the frequency, and whether a lower-frequency signal can pass through the same samples?


In [ ]:
sampling_signal_controls = signal_sampling_widget()

## 2. Interpolation in 1D

Interpolation fills in values between samples. Nearest, linear, and cubic interpolation make different assumptions about what happens between known points.

Exploration missions:

- Which method creates discontinuities?
- Which method changes slope abruptly at sample locations?
- When does the interpolated curve look convincing even if it is wrong?

In [ ]:
interp_1d_controls = interpolation_1d_widget()

## 3. Interpolation in 2D

In images, interpolation choices become visible as artifacts. Zoom into different crops and compare how each method treats edges, texture, and repeated patterns.

We did not cover Lanczos interpolation in class, but I recommend this 5-minute graphics video by Steve Seitz: [Lanczos interpolation](https://www.youtube.com/watch?v=Xj129kA3Ci0).

Exploration missions:

- What kinds of artifacts are created by each method? Can you name the effect?
- Which areas in the image are most susceptible, and which are less susceptible?
- How does the zoom factor change what you notice?
- Which method would you choose for a natural image? Which for a binary label, mask, or segmentation map?


In [ ]:
img = load_image("shinjuku-tokyo.jpg")
interp_2d_controls = interpolation_2d_widget(img)

## 4. Derivatives, Noise, and Smoothing

A derivative measures local change. Along the red slice, image intensity becomes a 1D function $I(t)$. The derivative $\frac{dI}{dt}$ is the local slope of that function: flat regions have small slope, bright-to-dark or dark-to-bright transitions have large positive or negative slope.

Move the slice and compare the image, the gradient magnitude, and the intensity and derivative profiles. The red markers show maximum values along the current slice; the green markers show minimum values.

Exploration missions:

- Move the slice across smooth regions and strong boundaries.
- Rotate the slice by 180 degrees. What changes in the intensity plot? What changes in the derivative plot?
- Can you infer which direction is increasing $t$ along the slice from the sign of the derivative?
- What does a strong positive or negative derivative mean along the slice direction?
- Use the slice position and angle to search for the strongest local change in the entire image.
- Can you find angles and offsets where the eyes produce the maximum derivative along the slice?
- Can you make the hairline produce the minimum derivative?
- Can you make the collar produce the maximum derivative?

In [ ]:
img = load_image("messi.jpg")
clean_derivative_controls = derivative_clean_widget(img)

After searching with the slice tool, commit to an answer: where is the largest positive gradient in the image, and what direction does it point? Mark your guess below, then click reveal to compare it with the computed answer.

In [ ]:
gradient_guess_controls = derivative_gradient_reveal_widget(img)

Now add noise in a separate experiment using the same image. Noise means that the measured pixel value is not exactly the true image value. This can happen for many reasons: electronics in the camera sensor can add small random fluctuations, and light itself arrives in discrete photons, so even a perfectly still scene is measured with some randomness.

Compare the original image, the noisy image, and the blurred noisy image. The bottom row compares their intensity profiles and derivative profiles along the same slice.

Exploration missions:

- Add noise. What happens to the slice profile and its derivative?
- Increase smoothing. Which derivative peaks disappear first?
- When does the blurred derivative become closer to the original derivative than the noisy derivative?
- Which parts of the image show the biggest change in the derivative plot after noise is added? Try slices through smooth regions, textured regions, and strong boundaries. What do you notice?

In [ ]:
noise_derivative_controls = derivative_noise_widget(img)

## 5. Directional Derivatives

A directional derivative asks: how quickly does the image change if we move in this direction? 

The cell below lets you explore directional derivatives. The derivative map shows the derivative at every pixel in the image along the selected direction. The polar plot shows the derivative along all directions for one chosen pixel.

Exploration missions:

- Move the point onto a strong edge. What do you notice about the direction of the gradient?
- Which direction gives a response close to zero? How is that direction related to the gradient direction?
- How does the strongest derivative direction relate to the visible edge direction?
- What happens at corners or textured regions?


In [ ]:
%matplotlib widget
img = load_image("messi.jpg")
directional_controls = directional_derivative_widget(img)

## 6. Canny Edge Detector as a System

Canny-style edge detection is not just one operation. It first estimates image gradients, then uses both gradient magnitude and gradient direction. The direction is used for non-maximum suppression (NMS): a pixel is kept only if its gradient magnitude is locally largest along the gradient direction. Hysteresis thresholds then decide which thin responses become edges.

Click on the gradient magnitude image. The cyan arrow shows the local gradient direction, and the 1D plot samples gradient magnitude along that direction. The center point is the clicked pixel; the two side points are the interpolated values that non-maximum suppression compares against.

Exploration missions:

- Find settings where many tiger stripes survive as final edges while minimizing the background.
- Can you make whiskers or fine fur texture survive? What else appears when you do this?
- Click on a stripe boundary and on the tiger’s outer contour. How do their 1D NMS profiles differ?
- Increase `smooth` until nearby stripes merge in the gradient magnitude image. What happens after NMS?
- Keep `smooth` fixed and raise `high`. Which edges disappear first: stripes, fur texture, whiskers, or the outer contour?
- Use the mouse to click and keyboard to move around and see the gradient orientation and the NMS profile. Explore which pixels get surpressed. 

In [ ]:
%matplotlib widget
img = load_image("tiger.jpg")
canny_controls = canny_widget(img)

## 7. Image Pixels in Chromaticity Space

RGB values are coordinates for a display. XYZ gives a standardized way to describe color, and xy chromaticity removes overall brightness:

$$
x = \frac{X}{X+Y+Z}, \qquad y = \frac{Y}{X+Y+Z}.
$$

The plot below shows image pixels in xy chromaticity space. The triangle is the sRGB display gamut, and the curved boundary is an approximate visible-color boundary.

Exploration missions:

- Without labels, can you identify which corner of the sRGB triangle is red, green, and blue? 
- Where do gray, white, and black image regions appear in xy space?
- Which parts of the image create points far from the white point?
- Without labels, can you identify which corner of the sRGB triangle is red, green, and blue? What image pixels helped you decide?
- Increase `min brightness`. Which colors disappear first? Are they gone from the image, or are they mostly from darker regions?
- Change `shown pixels`. Does the shape of the chromaticity cloud change, or only how densely it is drawn?
- Are most natural-image pixels near the boundary or inside the triangle?


In [ ]:
img = load_image("tucan.jpg")
chromaticity_controls = image_chromaticity_widget(img)

## 8. Color Constancy and White Balance

White balance tries to undo the color of the illumination. Different methods make different assumptions about what should look neutral in the scene.

Exploration missions:

- Switch from `regular` to `warm cast`, `cool shade`, and `underwater-like`. What changes in the corrected image? What changes in the RGB gains?
- What does it mean when a channel gain is greater than 1, less than 1, or close to 1?
- Compare `gray world`, `brightest`, and `patch` on the same image. Which correction looks most plausible, and where does it fail? What does `gray world` assume about the image?
- Move the patch over a white, gray, colored, shadowed, and bright region. Which patches make the correction better, and which make it worse?
- Find a setting where the correction removes the color cast but introduces a new color problem somewhere else.

In [ ]:
base = load_image("piazza.jpg")
images = {
    "regular": base,
    "warm cast": apply_color_cast(base, gains=(1.35, 1.05, 0.72)),
    "cool shade": apply_color_cast(base, gains=(0.75, 0.95, 1.35)),
    "underwater-like": simulate_underwater(base),
}
white_balance_controls = white_balance_widget(images)

## 9. Histogram Stretching and Gamma

The tone curve shows how each brightness value in the input is mapped to a brightness value in the output. The dashed line $y=x$ is the identity mapping: every input value stays unchanged. Moving the black point, white point, and gamma changes the shape of this curve.

Exploration missions:

- Start from `black = 0`, `white = 1`, and `gamma = 1`. What does the tone curve do to every input value?
- Move the black point just past the left edge of the histogram. Which dark regions become pure black?
- Move the white point just before the right edge of the histogram. Which bright regions become pure white?
- Put either border inside the main histogram mass. What visual detail is lost, and where does the tone curve become flat?
- After choosing black and white points, change `gamma`. Which regions change most without becoming fully black or white?
- Find settings that increase contrast but still look natural. Where are the two borders relative to the histogram?


In [ ]:
img = load_image("pasadena.jpg")
gamma_controls = gamma_tone_widget(img)

# Homework 3: 5.00 marks

Complete **both parts** and submit **one PDF report**. Part A is worth **3.75 marks**. Part B is a short experiment using
your own phone photographs, worth **1.25 marks**.

For every task, explain **what you did, why you chose it, and what you found**,
with the requested measurements and images. 

The supplied asset files are in **`homework3_assets/` beside this notebook**.

| Task | Assessed work | Marks |
|---|---|---:|
| A1 | Character colour correction | 1.00 |
| A2 | Find the numbers | 1.00 |
| A3 | Finding orientation | 1.00 |
| A4 | Putting it together | 0.50 |
| A5 | Evaluation and comparison | 0.25 |
| B1 | Phone captures and observations | 0.25 |
| B2 | Global AWB | 0.50 |
| B3 | Gray card white balancing | 0.50 |
| **Total** | | **5.00** |

## Part A: Character and canvas — 3.75 marks

You are given **a small character image and a separate background canvas**. The character image
has an artificial colour cast. The canvas's colour pattern
is tilted relative to a reference image which has the right color orientation. The canvas also contains a gray patch containing a pair of numbers.

Your goal is to correct the character's colours, recover the pair of numbers, upsample both
files, and assemble the final desired image. The canvas must be rotated to match the
reference's colour-pattern orientation. Apply that **same rotation** to the
character image, then paste it onto the rotated canvas at the recovered coordinate.

**Include this part in your homework PDF. No starter code is supplied, and no code (except for pseudocode for B3) is required to be submitted.** You may
use reuse any code provided in the notebook, accompanying helper modules, or write your own. 

Explain your choices, measurements, and results with supporting images or figures. You **DO NOT** need to submit your code for Part A.

### Files provided

The supplied source and reference images are NumPy arrays. Shapes are listed as
**(height, width, channels)**; image dimensions elsewhere are **width x height**.

| File | Contents | Array shape |
|---|---|---|
| `homework3_assets/character.npy` | Colour-shifted image of the character with transparent background | (48, 48, 4) |
| `homework3_assets/character_reference.npy` | Expected character colours before enlargement or rotation, with the same layout and transparency | (48, 48, 4) |
| `homework3_assets/canvas.npy` | The original background file | (120, 160, 3) |
| `homework3_assets/orientation_reference.npy` | The correct color orientation reference | (96, 96, 3) |
| `homework3_assets/desired_output.npz`, key `image` | Desired final RGB output, for numerical comparison | (540, 592, 3) |
| `homework3_assets/asset_guide.png` | Annotated views of the asset files | Preview only |
| `homework3_assets/targets/` | Outcome figures and details to inspect | Preview only |

Load the `.npy` files with `numpy.load`. All four `.npy` files contain `float64` values in
**[0, 1]**, with RGB channel order. The two character arrays have a fourth, alpha channel:
0 is transparent and 1 is opaque. No division by 255 is needed. The annotated
PNG figures are previews; use the arrays for all processing and measurements.
The `image` array inside `desired_output.npz` also contains `float64` values in
[0, 1]. Use this reference to evaluate your reconstruction.

![Supplied assets and labelled regions](homework3_assets/asset_guide.png)

Coordinates start at zero at the top-left: x increases rightward and y downward.
The rectangles below include the lower bound and exclude the upper bound.

- **Calibration patch, in `character.npy`:** x from 22 to 26, y from 30 to 34.
  This 4 x 4 chest patch was neutral before the colour cast. Its original RGB
  chromaticity was r = g = b = 1/3, where r = R/(R+G+B), and similarly for g and b.
- **Hidden-coordinate region, in `canvas.npy`:** x from 94 to 156, y from 66 to 118.
  This is the magenta rectangle in the guide. Inspect it in the original canvas,
  before enlargement or rotation.

Three positive RGB gains were applied to every nontransparent pixel of the
character. You do not need to apply color correction to the canvas

### A1. Character colour correction — 1.00 marks

Estimate a colour correction from the supplied information and apply it to
`character.npy` so that its colours match `character_reference.npy`. Retain its
transparency. Show your corrected character, report the parameters you estimated,
and explain your method with supporting measurements or figures.

For full marks, you'd need to use only the calibration patch information provided and the
corresponding patch in the reference character image. 

![Supplied character and expected character colours](homework3_assets/targets/01_colour_goal.png)

### A2. Find the numbers — 1.00 marks

In the original `canvas.npy`, the gray region is **x from 94 to 156** and
**y from 66 to 118**, including each lower bound and excluding each upper
bound (zero-based column and row coordinates).

The gray patch in `canvas.npy` contains two three-digit
values, one above the other. Experiment with different image operations introduced in this notebook and prior labs, and find the operation that
produces **maximum readability**: both values should be clearly distinguishable from the surrounding variation.

Hint: You can maybe see the variations by eye, but how can you maximize this information? What operation would accentuate this the most?

The **upper value is x** and the **lower value is y**. Together they specify
the top-left corner of the **entire rotated character layer** in the **final
canvas**, using zero-based (column, row) coordinates. They already refer to the
enlarged, rotated output: do not scale or rotate the coordinate again.

Report (x, y) and show your most readable result.

Explain what operation you chose, why it works, and why it gives greater
readability than the other operations you tried.

![](homework3_assets/targets/03_number_goal.png)

### A3. Finding orientation — 1.00 marks

Determine the rotation that aligns the canvas's colour pattern with the pattern
shown in `orientation_reference.npy`. The required rotation is less than 40 degrees in magnitude.
Note that the reference is just for the "color orientation", your output should not be cropped.

For full marks: your method should avoid colour matching -- do not find pixels with the same
colour in the input and reference images and compute the angle from those matches.

Report the angle (specify whether it is CW or CCW), and explain how you found it. You will apply this rotation to **both files in A4**.

![Orientation goal ](homework3_assets/targets/02_orientation_goal.png)

### A4. Putting it together — 0.50 marks

Use your color corrected character, the supplied canvas, the angle from A3, and the
coordinate from A2:

1. Upsample `character.npy` **3x to 144 x 144** and `canvas.npy` **3x to 480 x 360**.
2. Rotate each enlarged image about its own centre by the **same angle and
   direction** from A3. Render the rotated canvas in a **592 x 540** frame
   and the rotated character in a **194 x 194** frame, keeping each image centred
   in its frame. 
3. Place the top-left of the entire **194 x 194 character layer** at the decoded
   **(x, y)** in the final canvas. This includes the character layer's transparent
   padding; so you do not need crop it to the visible character first. You can use white for filling the
   background outside the rotated canvas.

Array element [y, x] has pixel-centre coordinates (x, y). For a W x H image,
the rotation centre is ((W-1)/2, (H-1)/2).

Your output should closely match the desired result shown below. Show the two
enlarged images **before rotation** and your final composition. Explain the
operations you chose and support your explanation with detail views or plots.

![](homework3_assets/targets/04_enlargement_goal.png)

The white box marks the strip averaged across for the example background profile.

![Desired output (desired_output.npz)](homework3_assets/targets/04_composition_goal.png)

### A5. Evaluation and comparison — 0.25 marks

Compare your final RGB array I with the supplied reference T. Report the mean
absolute error, **MAE = mean(abs(I - T))**, and include a per-pixel residual map
**D[y, x] = mean(abs(I[y, x, :] - T[y, x, :]))**, where the latter mean is over
RGB channels. For a perfect match, this map will contain zeros or values very
close to zero throughout. Include a colour bar and a short explanation of where
differences occur, if any. Show your own reconstructed result.

## Part B: Auto White Balancing — 1.25 marks

Explore what your phone does automatically when you capture an image. Submit your results in the homework PDF; no notebook or full source-code submission is required. You can provide pseudcode for your white balancing algorithm.

### B1. Phone captures and observations — 0.25 marks

Arrange some coloured objects with a matte white or neutral-gray object, such
as a sheet of paper. Take **two original photographs of the same arrangement
under visibly different lighting**, for example daylight and a warm indoor
lamp. Keep the framing similar. Use your phone's normal camera with automatic
white balance. No special camera app or RAW capture is required for this task.

Show both photographs and briefly describe the lighting and camera/app used.
Mark a small region on the neutral object in each photograph. Average the RGB
values in that region, then divide the three means by their sum to obtain its
RGB chromaticity. Report the two triples.

How much does the neutral object's saved colour change between photographs?
What do you think the phone is doing to compensate for the lighting? Support
your explanation with the photographs and measurements. Note you don't have to find the exact algorithm the camera uses, just hypothesize based on what you found in your captures.

### B2. Global AWB — 0.50 marks

In a short algorithm description or pseudocode, propose how to white-balance an
image **without a user selecting a neutral reference**. State how you would estimate
the colour cast from the image and how that estimate would change its colours (establish your own assumption of what the color distribution should be). 

Explain the assumption behind your method and give one scene where it could fail. You don't need to implement anything for this task.

### B3. Gray card white balancing — 0.50 marks

Gray-card white balancing is standard practice in photography and
cinematography, especially when colour and lighting are controlled manually
rather than left to automatic white-balance algorithms. See
[this short video on gray-card white balancing](https://www.youtube.com/shorts/HQVRBP9TtHQ).

Choose **one** of your two original photographs. Apply a [warming/cooling filter on a captured image (example video for iPhone)](https://www.youtube.com/watch?v=FEQdSeAxKpw) to simulate a different lighting condition. 

![Example: original photograph on the left and the same photograph with a simulated colour cast on the right](homework3_assets/b3_white_balance_example.png)

Write a short piece of code that accepts the edited image and the location of
neutral color (e.g. sheet of paper), estimates one set of RGB channel gains from that
reference, and applies them to the **entire image**. 

Provide pseudocode of your algorithm in the PDF report.


Show the **original, edited, and corrected** photographs side by side, with the
calibration region marked. Report the three gains and the region's RGB
chromaticity in all three images. Include only the main white-balancing code
snippet, showing how you estimate and apply the gains.

Explain what improved and what differences remain, if any. 

Since usually the usual photo editing app change more than channel gains from a single slider, so exact recovery of the
original photograph is not required for full marks here.

## Submission checklist

Submit **one PDF report** containing both parts, with your answers labelled
**A1–A5 and B1–B3**. For each task, explain **what you did, why you chose it,
and what you found**, supported by the requested measurements and figures.

### Part A report

- **A1. Character colour correction:** Your corrected character with its transparency retained, the estimated correction parameters, and supporting measurements and reasoning. For full marks, base the estimate on the supplied calibration patch and its corresponding patch in the reference character.
- **A2. Find the numbers:** The recovered **(x, y)** coordinate, your most readable image of the hidden numbers, and an explanation of your chosen operation and why it gives greater readability than the other operations you tried.
- **A3. Finding orientation:** The estimated angle, whether it is **CW or CCW**, and the measurements and reasoning used to obtain it. For full marks, avoid finding same-colour pixel matches between the input and reference to compute the angle.
- **A4. Putting it together:** Both enlarged images **before rotation**, your own final composition, and an explanation of your operations supported by detail views or plots. Use the dimensions, rotation convention, and placement specified in A4.
- **A5. Evaluation and comparison:** The numerical **MAE = mean(abs(I - T))**, a **full residual map with a colour bar**, and a short explanation of where differences occur, if any. Compute the map as **D[y, x] = mean(abs(I[y, x, :] - T[y, x, :]))**, averaging over RGB channels. A perfect match gives zeros or values very close to zero throughout.

No code submission is required for Part A.

### Part B report

- **B1. Phone captures and observations:** Your **two original photographs** of the same arrangement under different lighting, a description of the lighting and camera/app used, marked neutral regions, and their measured RGB chromaticities. Discuss the observed changes and what you think the phone's automatic white balance is doing, using your photographs and measurements as evidence.
- **B2. Global AWB:** Your proposed algorithm description or pseudocode for estimating and correcting the colour cast **without a user-selected neutral reference**, its assumption, and one scene where it could fail. No implementation is required for B2.
- **B3. Gray card white balancing:** Your **original, edited, and corrected photographs side by side**, with the calibration region marked. Report the three gains and the region's RGB chromaticity in all three images. Include your main white-balancing algorithm as a **short code snippet or pseudocode**, showing how you estimate the gains and apply them to the entire image. Explain what improved and what differences remain, if any; exact recovery of the original photograph is not required for B3.

Embed all photographs, output images, and figures in the PDF so no separate
image files are required. Make figure labels, plot axes, colour bars, and
numerical results large enough to read. You do not need to submit the supplied
assets, helper modules, or original lab ZIP separately.

You may also submit this notebook with your code included, but it is **optional**.
